In [10]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import os

RAW_VIDEO_PATH = "/content/drive/MyDrive/SignNumberModel/raw_vedios"

# Check whether the folder exists
print("Folder exists:", os.path.exists(RAW_VIDEO_PATH))

# Show class folders
classes = sorted([f for f in os.listdir(RAW_VIDEO_PATH) if os.path.isdir(os.path.join(RAW_VIDEO_PATH, f))])

print("Number of classes found:", len(classes))
print("First 20 class names:", classes[:20])

Folder exists: True
Number of classes found: 88
First 20 class names: ['1', '10', '100', '103', '105', '109', '11', '110', '12', '124', '13', '130', '135', '14', '140', '15', '150', '16', '160', '165']


In [12]:
# Count video files inside each class folder
for class_name in classes[:10]:   # first 10 classes only
    class_path = os.path.join(RAW_VIDEO_PATH, class_name)
    files = [f for f in os.listdir(class_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]
    print(f"{class_name}: {len(files)} videos")

1: 11 videos
10: 10 videos
100: 14 videos
103: 13 videos
105: 14 videos
109: 15 videos
11: 10 videos
110: 16 videos
12: 12 videos
124: 16 videos


In [13]:
# Step 2.1 - Install needed libraries
# mediapipe = hand landmark extraction
# opencv = read videos
# tensorflow = later for model training
# Install correct MediaPipe version
!pip install mediapipe==0.10.20 opencv-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 81.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.2/81.2 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 10.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 4.25.9 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.

In [1]:
import mediapipe as mp

print(mp.__version__)  # check version

0.10.20


In [2]:
# Step 2.2 - Import libraries
import os
import cv2
import numpy as np
import mediapipe as mp

In [3]:
# Step 2.3 - Set input and output paths
# raw video folder = your original dataset
# landmark folder = extracted landmark data will be saved here

RAW_VIDEO_PATH = "/content/drive/MyDrive/SignNumberModel/raw_vedios"
LANDMARK_PATH = "/content/drive/MyDrive/SignNumberModel/landmark_data"

# Create output folder if it does not exist
os.makedirs(LANDMARK_PATH, exist_ok=True)

print("Raw video path exists:", os.path.exists(RAW_VIDEO_PATH))
print("Landmark output path exists:", os.path.exists(LANDMARK_PATH))

Raw video path exists: True
Landmark output path exists: True


In [4]:
# Step 2.4 - Load class names again
# This gets all class folders from raw videos

classes = sorted([
    f for f in os.listdir(RAW_VIDEO_PATH)
    if os.path.isdir(os.path.join(RAW_VIDEO_PATH, f))
])

print("Total classes:", len(classes))
print("Sample classes:", classes[:10])

Total classes: 88
Sample classes: ['1', '10', '100', '103', '105', '109', '11', '110', '12', '124']


In [5]:
# Step 3.1 - Setup MediaPipe Hands
mp_hands = mp.solutions.hands

hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

print("MediaPipe Hands initialized successfully")

MediaPipe Hands initialized successfully


In [6]:
# Step 3.2 - Extract landmarks from one frame

def extract_landmarks(frame):
    # Convert BGR (OpenCV) → RGB (MediaPipe needs RGB)
    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process frame
    results = hands.process(image)

    if results.multi_hand_landmarks:
        hand = results.multi_hand_landmarks[0]  # take first hand

        landmarks = []
        for lm in hand.landmark:
            # x, y, z for each of 21 points → total 63
            landmarks.extend([lm.x, lm.y, lm.z])

        return np.array(landmarks)

    else:
        # if no hand detected → return zeros
        return np.zeros(63)

In [7]:
# Step 3.3 - Test extraction on one video

# pick first class
test_class = classes[0]

test_class_path = os.path.join(RAW_VIDEO_PATH, test_class)

# get video list
video_files = [f for f in os.listdir(test_class_path) if f.endswith('.mp4')]

# pick first video
video_path = os.path.join(test_class_path, video_files[0])

print("Testing video:", video_path)

cap = cv2.VideoCapture(video_path)

sequence = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    landmarks = extract_landmarks(frame)
    sequence.append(landmarks)

cap.release()

sequence = np.array(sequence)

print("Sequence shape:", sequence.shape)

Testing video: /content/drive/MyDrive/SignNumberModel/raw_vedios/1/number1_vedio1.mp4
Sequence shape: (79, 63)


In [8]:
# Step 4.1 - Convert any sequence to fixed 40 frames

SEQ_LEN = 40   # final number of frames for each video

def resize_sequence(sequence, seq_len=SEQ_LEN):
    # If video has more frames than needed
    if len(sequence) > seq_len:
        # Pick frames evenly
        indices = np.linspace(0, len(sequence) - 1, seq_len).astype(int)
        sequence = sequence[indices]

    # If video has fewer frames than needed
    elif len(sequence) < seq_len:
        # Add zero frames at the end
        padding = np.zeros((seq_len - len(sequence), 63))
        sequence = np.vstack((sequence, padding))

    return sequence

In [9]:
# Step 4.2 - Test fixed-length conversion

fixed_sequence = resize_sequence(sequence)

print("Old shape:", sequence.shape)
print("New shape:", fixed_sequence.shape)

Old shape: (79, 63)
New shape: (40, 63)


In [10]:
# Step 5.1 - Create label mapping (class → number)

label_map = {label: idx for idx, label in enumerate(classes)}

print("Total classes:", len(label_map))
print("Sample mapping:", list(label_map.items())[:5])

Total classes: 88
Sample mapping: [('1', 0), ('10', 1), ('100', 2), ('103', 3), ('105', 4)]


In [ ]:
# Step 5.2 - Extract landmarks from ALL videos

X = []   # store sequences (data)
y = []   # store labels

for class_name in classes:
    class_path = os.path.join(RAW_VIDEO_PATH, class_name)

    # get ALL video formats (mp4, mov, avi, mkv)
    video_files = [
        f for f in os.listdir(class_path)
        if f.lower().endswith(('.mp4', '.mov', '.avi', '.mkv'))
    ]

    print(f"Processing class: {class_name} ({len(video_files)} videos)")

    for video_file in video_files:
        video_path = os.path.join(class_path, video_file)

        cap = cv2.VideoCapture(video_path)
        sequence = []

        # read video frame by frame
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # extract 63 features from one frame
            landmarks = extract_landmarks(frame)
            sequence.append(landmarks)

        cap.release()

        sequence = np.array(sequence)

        # skip empty videos
        if len(sequence) == 0:
            continue

        # convert to fixed length (40, 63)
        sequence = resize_sequence(sequence)

        # save data
        X.append(sequence)
        y.append(label_map[class_name])

Processing class: 1 (11 videos)
Processing class: 10 (10 videos)
Processing class: 100 (14 videos)
Processing class: 103 (13 videos)
Processing class: 105 (14 videos)
Processing class: 109 (15 videos)
Processing class: 11 (10 videos)
Processing class: 110 (16 videos)
Processing class: 12 (12 videos)
Processing class: 124 (16 videos)
Processing class: 13 (10 videos)
Processing class: 130 (15 videos)
Processing class: 135 (15 videos)
Processing class: 14 (10 videos)
Processing class: 140 (16 videos)
Processing class: 15 (10 videos)
Processing class: 150 (16 videos)
Processing class: 16 (10 videos)
Processing class: 160 (15 videos)
Processing class: 165 (16 videos)
Processing class: 17 (10 videos)
Processing class: 170 (17 videos)
Processing class: 18 (10 videos)
Processing class: 19 (10 videos)
Processing class: 190 (19 videos)
Processing class: 199 (20 videos)
Processing class: 2 (10 videos)
Processing class: 20 (8 videos)
Processing class: 200 (24 videos)
Processing class: 210 (24 vide

In [ ]:
# Import numpy again
import numpy as np

In [ ]:
# Step 5.3 - Convert to numpy arrays

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1085, 40, 63)
y shape: (1085,)


In [ ]:
# Step 6.1 - Import
from sklearn.model_selection import train_test_split

In [ ]:
# Step 6.2 - Split dataset

# 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,        # keep class balance
    random_state=42
)

# 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (759, 40, 63)
X_val: (163, 40, 63)
X_test: (163, 40, 63)


In [ ]:
# Step 7.1 - Import tensorflow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, BatchNormalization

In [ ]:
# Step 7.2 - Build model

num_classes = len(classes)

model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(40, 63)),
    BatchNormalization(),
    Dropout(0.3),

    Bidirectional(LSTM(64)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 40, 256)        │       196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 40, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 88)             │        11,352 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,360 (1.49 MB)

 Trainable params: 389,592 (1.49 MB)

 Non-trainable params: 768 (3.00 KB)

In [ ]:
# Step 8 - Train

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=32
)

Epoch 1/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 16s 212ms/step - accuracy: 0.0198 - loss: 4.9322 - val_accuracy: 0.0307 - val_loss: 4.4724
Epoch 2/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 236ms/step - accuracy: 0.0382 - loss: 4.5545 - val_accuracy: 0.0184 - val_loss: 4.4507
Epoch 3/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 4s 174ms/step - accuracy: 0.0435 - loss: 4.4116 - val_accuracy: 0.0123 - val_loss: 4.4172
Epoch 4/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 213ms/step - accuracy: 0.0553 - loss: 4.1782 - val_accuracy: 0.0307 - val_loss: 4.3478
Epoch 5/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 6s 242ms/step - accuracy: 0.0540 - loss: 4.1040 - val_accuracy: 0.0245 - val_loss: 4.3151
Epoch 6/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 213ms/step - accuracy: 0.0751 - loss: 3.9238 - val_accuracy: 0.0552 - val_loss: 4.2235
Epoch 7/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 9s 172ms/step - accuracy: 0.1014 - loss: 3.7346 - val_accuracy: 0.0368 - val_loss: 4.2135
Epoch 8/50
24/24 ━━━━━━━━━━━━━━━━━━━━ 5s 225ms/step - accuracy: 0.0922 - loss: 3.6918 - val_accuracy: 

In [ ]:
# Step 9.1 - Evaluate on test data

test_loss, test_acc = model.evaluate(X_test, y_test)

print("Test Accuracy:", test_acc)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.5337 - loss: 1.4255
Test Accuracy: 0.5337423086166382


In [ ]:
# Save current model
model.save("/content/drive/MyDrive/SignNumberModel/sign_model_current.keras")

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        verbose=1
    ),
    ModelCheckpoint(
        "/content/drive/MyDrive/SignNumberModel/best_sign_model.keras",
        monitor='val_accuracy',
        save_best_only=True
    )
]

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=16,
    callbacks=callbacks
)

Epoch 1/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 9s 180ms/step - accuracy: 0.6350 - loss: 1.1795 - val_accuracy: 0.2515 - val_loss: 3.4504 - learning_rate: 0.0010
Epoch 2/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 5s 108ms/step - accuracy: 0.6206 - loss: 1.1436 - val_accuracy: 0.2393 - val_loss: 3.2780 - learning_rate: 0.0010
Epoch 3/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 116ms/step - accuracy: 0.5837 - loss: 1.2947 - val_accuracy: 0.1104 - val_loss: 4.9862 - learning_rate: 0.0010
Epoch 4/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s 147ms/step - accuracy: 0.6337 - loss: 1.0967 - val_accuracy: 0.4540 - val_loss: 1.9143 - learning_rate: 0.0010
Epoch 5/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 5s 107ms/step - accuracy: 0.6509 - loss: 1.0013 - val_accuracy: 0.3129 - val_loss: 3.2253 - learning_rate: 0.0010
Epoch 6/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s 147ms/step - accuracy: 0.6983 - loss: 0.9641 - val_accuracy: 0.2945 - val_loss: 3.5370 - learning_rate: 0.0010
Epoch 7/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.6970 - loss: 0.9327 - 

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

num_classes = len(classes)

model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(40, 63)),
    BatchNormalization(),
    Dropout(0.4),

    Bidirectional(LSTM(64)),
    BatchNormalization(),
    Dropout(0.4),

    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.2),

    Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_2 (Bidirectional) │ (None, 40, 256)        │       196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 40, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 88)             │         5,720 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 392,984 (1.50 MB)

 Trainable params: 392,216 (1.50 MB)

 Non-trainable params: 768 (3.00 KB)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1),
    ModelCheckpoint(
        "/content/drive/MyDrive/SignNumberModel/best_sign_model.keras",
        monitor='val_accuracy',
        save_best_only=True
    )
]

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=16,
    callbacks=callbacks
)

Epoch 1/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 19s 167ms/step - accuracy: 0.0184 - loss: 4.7682 - val_accuracy: 0.0184 - val_loss: 4.4702 - learning_rate: 5.0000e-04
Epoch 2/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s 107ms/step - accuracy: 0.0119 - loss: 4.5524 - val_accuracy: 0.0061 - val_loss: 4.4599 - learning_rate: 5.0000e-04
Epoch 3/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 11s 124ms/step - accuracy: 0.0145 - loss: 4.5050 - val_accuracy: 0.0245 - val_loss: 4.4337 - learning_rate: 5.0000e-04
Epoch 4/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 5s 111ms/step - accuracy: 0.0356 - loss: 4.4595 - val_accuracy: 0.0429 - val_loss: 4.4067 - learning_rate: 5.0000e-04
Epoch 5/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s 157ms/step - accuracy: 0.0303 - loss: 4.4298 - val_accuracy: 0.0429 - val_loss: 4.3752 - learning_rate: 5.0000e-04
Epoch 6/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 5s 112ms/step - accuracy: 0.0316 - loss: 4.4174 - val_accuracy: 0.0613 - val_loss: 4.3710 - learning_rate: 5.0000e-04
Epoch 7/60
48/48 ━━━━━━━━━━━━━━━━━━━━ 7s 140ms/step - accuracy

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_acc)

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - accuracy: 0.4356 - loss: 1.7606
Test Accuracy: 0.43558281660079956


In [ ]:
# Step 1.1 - Import Counter
# Counter helps count how many samples are in each class
from collections import Counter
import numpy as np

# Step 1.2 - Count samples in full dataset and splits
full_counts = Counter(y)
train_counts = Counter(y_train)
val_counts = Counter(y_val)
test_counts = Counter(y_test)

# Step 1.3 - Print basic summary
print("Total classes:", len(full_counts))
print("Total samples:", len(y))
print("Minimum samples in one class:", min(full_counts.values()))
print("Maximum samples in one class:", max(full_counts.values()))
print("Average samples per class:", round(np.mean(list(full_counts.values())), 2))

# Step 1.4 - Show classes with very low samples
print("\nClasses with 10 or fewer total samples:")
for cls, count in sorted(full_counts.items(), key=lambda x: x[1]):
    if count <= 10:
        print(f"Class {cls}: {count}")

# Step 1.5 - Check smallest counts in train/val/test
print("\nMinimum samples per class in TRAIN split:", min(train_counts.values()))
print("Minimum samples per class in VAL split:", min(val_counts.values()))
print("Minimum samples per class in TEST split:", min(test_counts.values()))

# Step 1.6 - Show first 10 class counts from each split
print("\nFirst 10 TRAIN class counts:")
for i, (cls, count) in enumerate(sorted(train_counts.items())[:10]):
    print(f"Class {cls}: {count}")

print("\nFirst 10 VAL class counts:")
for i, (cls, count) in enumerate(sorted(val_counts.items())[:10]):
    print(f"Class {cls}: {count}")

print("\nFirst 10 TEST class counts:")
for i, (cls, count) in enumerate(sorted(test_counts.items())[:10]):
    print(f"Class {cls}: {count}")

Total classes: 88
Total samples: 1085
Minimum samples in one class: 7
Maximum samples in one class: 24
Average samples per class: 12.33

Classes with 10 or fewer total samples:
Class 54: 7
Class 27: 8
Class 47: 8
Class 34: 9
Class 45: 9
Class 49: 9
Class 1: 10
Class 6: 10
Class 10: 10
Class 13: 10
Class 15: 10
Class 17: 10
Class 20: 10
Class 22: 10
Class 23: 10
Class 26: 10
Class 32: 10
Class 33: 10
Class 35: 10
Class 36: 10
Class 37: 10
Class 38: 10
Class 43: 10
Class 46: 10
Class 48: 10
Class 50: 10
Class 51: 10
Class 52: 10
Class 53: 10
Class 60: 10
Class 67: 10
Class 82: 10
Class 83: 10
Class 84: 10
Class 85: 10
Class 87: 10

Minimum samples per class in TRAIN split: 5
Minimum samples per class in VAL split: 1
Minimum samples per class in TEST split: 1

First 10 TRAIN class counts:
Class 0: 8
Class 1: 7
Class 2: 10
Class 3: 9
Class 4: 10
Class 5: 11
Class 6: 7
Class 7: 11
Class 8: 8
Class 9: 11

First 10 VAL class counts:
Class 0: 1
Class 1: 1
Class 2: 2
Class 3: 2
Class 4: 2
Class

In [ ]:
# Step 2.1 - Split into only TRAIN and VALIDATION
# This avoids the error from very small classes

from sklearn.model_selection import train_test_split
from collections import Counter

# 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,      # 20% validation
    stratify=y,          # keep class balance
    random_state=42
)

# Step 2.2 - Print shapes
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

# Step 2.3 - Check minimum samples per class
train_counts = Counter(y_train)
val_counts = Counter(y_val)

print("\nMinimum samples per class in TRAIN split:", min(train_counts.values()))
print("Minimum samples per class in VAL split:", min(val_counts.values()))

X_train shape: (868, 40, 63)
X_val shape: (217, 40, 63)
y_train shape: (868,)
y_val shape: (217,)

Minimum samples per class in TRAIN split: 6
Minimum samples per class in VAL split: 1


In [ ]:
# Step 3.1 - Create class weights
# This gives higher importance to classes with fewer samples

from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# Get all unique class labels from training data
unique_classes = np.unique(y_train)

# Compute balanced weights
weights = compute_class_weight(
    class_weight='balanced',
    classes=unique_classes,
    y=y_train
)

# Convert to dictionary format required by Keras
class_weights = dict(zip(unique_classes, weights))

# Print first 10 class weights
print("First 10 class weights:")
for i, (cls, w) in enumerate(class_weights.items()):
    if i < 10:
        print(f"Class {cls}: {w:.4f}")

First 10 class weights:
Class 0: 1.0960
Class 1: 1.2330
Class 2: 0.8967
Class 3: 0.9864
Class 4: 0.8967
Class 5: 0.8220
Class 6: 1.2330
Class 7: 0.7587
Class 8: 0.9864
Class 9: 0.7587


In [ ]:
# Step 3.2 - Check min and max class weights

all_weights = list(class_weights.values())

print("Minimum class weight:", min(all_weights))
print("Maximum class weight:", max(all_weights))

Minimum class weight: 0.5191387559808612
Maximum class weight: 1.643939393939394


In [ ]:
# Step 4.1 - Build BiLSTM model
# This model is a bit stronger and uses dropout to reduce overfitting

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout, BatchNormalization

num_classes = len(classes)

model = Sequential([
    # First BiLSTM layer
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(40, 63)),
    BatchNormalization(),
    Dropout(0.4),

    # Second BiLSTM layer
    Bidirectional(LSTM(64)),
    BatchNormalization(),
    Dropout(0.4),

    # Dense layers
    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(64, activation='relu'),
    Dropout(0.2),

    # Output layer
    Dense(num_classes, activation='softmax')
])

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Show model summary
model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_4 (Bidirectional) │ (None, 40, 256)        │       196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 40, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_5 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 88)             │         5,720 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 392,984 (1.50 MB)

 Trainable params: 392,216 (1.50 MB)

 Non-trainable params: 768 (3.00 KB)

In [ ]:
# Step 4.2 - Create callbacks
# EarlyStopping = stop if validation stops improving
# ReduceLROnPlateau = lower learning rate automatically
# ModelCheckpoint = save best model only

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        verbose=1
    ),
    ModelCheckpoint(
        "/content/drive/MyDrive/SignNumberModel/best_sign_model_weighted.keras",
        monitor='val_accuracy',
        save_best_only=True
    )
]

In [ ]:
# Step 4.3 - Train model with class weights
# class_weight helps smaller classes get more attention

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=60,
    batch_size=16,
    callbacks=callbacks,
    class_weight=class_weights
)

Epoch 1/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 18s 175ms/step - accuracy: 0.0150 - loss: 4.7493 - val_accuracy: 0.0184 - val_loss: 4.4727 - learning_rate: 5.0000e-04
Epoch 2/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 6s 110ms/step - accuracy: 0.0127 - loss: 4.5924 - val_accuracy: 0.0461 - val_loss: 4.4599 - learning_rate: 5.0000e-04
Epoch 3/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - accuracy: 0.0127 - loss: 4.5662 - val_accuracy: 0.0230 - val_loss: 4.4417 - learning_rate: 5.0000e-04
Epoch 4/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 145ms/step - accuracy: 0.0196 - loss: 4.4941 - val_accuracy: 0.0230 - val_loss: 4.4363 - learning_rate: 5.0000e-04
Epoch 5/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 108ms/step - accuracy: 0.0311 - loss: 4.4704 - val_accuracy: 0.0092 - val_loss: 4.4225 - learning_rate: 5.0000e-04
Epoch 6/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 8s 149ms/step - accuracy: 0.0265 - loss: 4.4377 - val_accuracy: 0.0276 - val_loss: 4.4199 - learning_rate: 5.0000e-04
Epoch 7/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 6s 112ms/step - accuracy

In [ ]:
# Step 5.1 - Evaluate model on validation data

val_loss, val_acc = model.evaluate(X_val, y_val)

print("Validation Accuracy:", val_acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 110ms/step - accuracy: 0.3641 - loss: 2.1934
Validation Accuracy: 0.3640553057193756


In [ ]:
# Step 1.1 - Normalize one sequence
# This function:
# 1. reshapes (40, 63) -> (40, 21, 3)
# 2. uses wrist landmark (point 0) as center
# 3. scales hand using max distance from wrist
# 4. returns back to (40, 63)

import numpy as np

def normalize_sequence(sequence):
    # Copy sequence to avoid changing original data directly
    seq = sequence.copy()

    # Reshape from (40, 63) to (40, 21, 3)
    seq = seq.reshape(seq.shape[0], 21, 3)

    normalized_frames = []

    for frame in seq:
        # Get wrist point (landmark 0)
        wrist = frame[0]

        # Move all landmarks so wrist becomes (0,0,0)
        centered = frame - wrist

        # Compute hand size using farthest landmark distance from wrist
        distances = np.linalg.norm(centered, axis=1)
        scale = np.max(distances)

        # Avoid divide-by-zero
        if scale > 0:
            centered = centered / scale

        # Flatten back to 63 features
        normalized_frames.append(centered.flatten())

    return np.array(normalized_frames)

In [ ]:
# Step 1.2 - Normalize full dataset
# Apply normalization to every sample in X

X_normalized = np.array([normalize_sequence(seq) for seq in X])

# Print shape to confirm
print("Old X shape:", X.shape)
print("New X_normalized shape:", X_normalized.shape)

Old X shape: (1085, 40, 63)
New X_normalized shape: (1085, 40, 63)


In [ ]:
# Step 1.3 - Check one sample before and after normalization
# This helps confirm values changed correctly

print("Before normalization (first 10 values):")
print(X[0][0][:10])

print("\nAfter normalization (first 10 values):")
print(X_normalized[0][0][:10])

Before normalization (first 10 values):
[ 4.53426778e-01  5.97200036e-01  3.50974396e-08  4.75107521e-01
  5.68550885e-01 -1.51178371e-02  4.90656406e-01  5.19140840e-01
 -2.75118556e-02  4.79310840e-01]

After normalization (first 10 values):
[ 0.          0.          0.          0.06696233 -0.0884847  -0.0466925
  0.11498604 -0.24109072 -0.0849722   0.07994455]


In [ ]:
# Step 2.1 - Split normalized data into train and validation
# We use X_normalized instead of X

from sklearn.model_selection import train_test_split
from collections import Counter

X_train_n, X_val_n, y_train_n, y_val_n = train_test_split(
    X_normalized, y,
    test_size=0.20,      # 20% validation
    stratify=y,          # keep class balance
    random_state=42
)

# Step 2.2 - Print shapes
print("X_train_n shape:", X_train_n.shape)
print("X_val_n shape:", X_val_n.shape)
print("y_train_n shape:", y_train_n.shape)
print("y_val_n shape:", y_val_n.shape)

# Step 2.3 - Check minimum samples per class
train_counts_n = Counter(y_train_n)
val_counts_n = Counter(y_val_n)

print("\nMinimum samples per class in TRAIN split:", min(train_counts_n.values()))
print("Minimum samples per class in VAL split:", min(val_counts_n.values()))

X_train_n shape: (868, 40, 63)
X_val_n shape: (217, 40, 63)
y_train_n shape: (868,)
y_val_n shape: (217,)

Minimum samples per class in TRAIN split: 6
Minimum samples per class in VAL split: 1


In [ ]:
# Step 3.1 - Build a fresh BiLSTM model for normalized data

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout, BatchNormalization

num_classes = len(classes)

model_n = Sequential([
    # First BiLSTM layer
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(40, 63)),
    BatchNormalization(),
    Dropout(0.3),

    # Second BiLSTM layer
    Bidirectional(LSTM(64)),
    BatchNormalization(),
    Dropout(0.3),

    # Dense layers
    Dense(128, activation='relu'),
    Dropout(0.3),

    Dense(num_classes, activation='softmax')
])

# Step 3.2 - Compile model
model_n.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Step 3.3 - Show model summary
model_n.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_6 (Bidirectional) │ (None, 40, 256)        │       196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 40, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_7 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 88)             │        11,352 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,360 (1.49 MB)

 Trainable params: 389,592 (1.49 MB)

 Non-trainable params: 768 (3.00 KB)

In [ ]:
# Step 4.1 - Create callbacks for better training

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks_n = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        verbose=1
    ),
    ModelCheckpoint(
        "/content/drive/MyDrive/SignNumberModel/best_sign_model_normalized.keras",
        monitor='val_accuracy',
        save_best_only=True
    )
]

In [ ]:
# Step 5.1 - Train model on normalized data

history_n = model_n.fit(
    X_train_n, y_train_n,
    validation_data=(X_val_n, y_val_n),
    epochs=60,
    batch_size=16,
    callbacks=callbacks_n
)

Epoch 1/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 17s 169ms/step - accuracy: 0.0242 - loss: 4.7538 - val_accuracy: 0.0461 - val_loss: 4.4371 - learning_rate: 5.0000e-04
Epoch 2/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 114ms/step - accuracy: 0.0668 - loss: 4.2517 - val_accuracy: 0.0507 - val_loss: 4.3144 - learning_rate: 5.0000e-04
Epoch 3/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 117ms/step - accuracy: 0.1198 - loss: 3.8209 - val_accuracy: 0.1244 - val_loss: 4.0505 - learning_rate: 5.0000e-04
Epoch 4/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 136ms/step - accuracy: 0.1613 - loss: 3.4231 - val_accuracy: 0.2212 - val_loss: 3.6689 - learning_rate: 5.0000e-04
Epoch 5/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 7s 123ms/step - accuracy: 0.2270 - loss: 3.1031 - val_accuracy: 0.2442 - val_loss: 3.3546 - learning_rate: 5.0000e-04
Epoch 6/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 12s 153ms/step - accuracy: 0.2719 - loss: 2.8376 - val_accuracy: 0.3272 - val_loss: 2.9071 - learning_rate: 5.0000e-04
Epoch 7/60
55/55 ━━━━━━━━━━━━━━━━━━━━ 6s 109ms/step - accurac

In [ ]:
# Step 6.1 - Evaluate normalized model

val_loss, val_acc = model_n.evaluate(X_val_n, y_val_n)

print("Validation Accuracy (Normalized Model):", val_acc)

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 151ms/step - accuracy: 0.8065 - loss: 0.6368
Validation Accuracy (Normalized Model): 0.8064516186714172


In [ ]:
# Step 6.2 - Predict classes

y_pred_n = model_n.predict(X_val_n)
y_pred_n = np.argmax(y_pred_n, axis=1)

print("Prediction shape:", y_pred_n.shape)

7/7 ━━━━━━━━━━━━━━━━━━━━ 3s 251ms/step
Prediction shape: (217,)


In [ ]:
# Step 6.3 - Manual accuracy

from sklearn.metrics import accuracy_score

acc = accuracy_score(y_val_n, y_pred_n)
print("Manual Accuracy:", acc)

Manual Accuracy: 0.8064516129032258


In [ ]:
# Step 6.4 - Detailed report

from sklearn.metrics import classification_report

print(classification_report(y_val_n, y_pred_n))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       1.00      1.00      1.00         2
           2       1.00      0.67      0.80         3
           3       1.00      1.00      1.00         3
           4       0.60      1.00      0.75         3
           5       1.00      1.00      1.00         3
           6       0.00      0.00      0.00         2
           7       1.00      1.00      1.00         3
           8       1.00      0.50      0.67         2
           9       1.00      0.67      0.80         3
          10       1.00      1.00      1.00         2
          11       0.67      0.67      0.67         3
          12       1.00      0.67      0.80         3
          13       0.50      0.50      0.50         2
          14       0.50      0.67      0.57         3
          15       1.00      0.50      0.67         2
          16       1.00      1.00      1.00         3
          17       1.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Step 7.1 - Find weak classes (low F1 score)

from sklearn.metrics import classification_report

report = classification_report(y_val_n, y_pred_n, output_dict=True)

weak_classes = []

for cls, metrics in report.items():
    if cls.isdigit():   # only class labels
        if metrics['f1-score'] < 0.6:   # threshold
            weak_classes.append((int(cls), metrics['f1-score'], metrics['support']))

# Sort by worst performance
weak_classes = sorted(weak_classes, key=lambda x: x[1])

print("Weak classes (class, f1-score, support):")
for cls in weak_classes[:15]:
    print(cls)

Weak classes (class, f1-score, support):
(0, 0.0, 2.0)
(6, 0.0, 2.0)
(22, 0.0, 2.0)
(23, 0.0, 2.0)
(54, 0.0, 1.0)
(68, 0.0, 2.0)
(30, 0.3333333333333333, 4.0)
(13, 0.5, 2.0)
(33, 0.5, 2.0)
(35, 0.5, 2.0)
(53, 0.5, 2.0)
(14, 0.5714285714285714, 3.0)
(69, 0.5714285714285714, 3.0)
(81, 0.5714285714285714, 3.0)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Step 8.1 - Augment sequence (add small noise)

import numpy as np

def augment_sequence(seq):
    # Add small random noise
    noise = np.random.normal(0, 0.01, seq.shape)
    return seq + noise

In [ ]:
# Step 8.2 - Augment weak classes only

X_aug = []
y_aug = []

# Weak class list (only class ids)
weak_class_ids = [cls for cls, _, _ in weak_classes]

for i in range(len(X_train_n)):
    seq = X_train_n[i]
    label = y_train_n[i]

    # Always keep original
    X_aug.append(seq)
    y_aug.append(label)

    # If weak class → add extra samples
    if label in weak_class_ids:
        # create 2 augmented versions
        for _ in range(2):
            new_seq = augment_sequence(seq)
            X_aug.append(new_seq)
            y_aug.append(label)

# Convert to numpy
X_aug = np.array(X_aug)
y_aug = np.array(y_aug)

print("Original train size:", X_train_n.shape)
print("Augmented train size:", X_aug.shape)

Original train size: (868, 40, 63)
Augmented train size: (1134, 40, 63)


In [ ]:
from collections import Counter

aug_counts = Counter(y_aug)

print("\nSample counts after augmentation (first 10):")
for i, (cls, count) in enumerate(sorted(aug_counts.items())[:10]):
    print(f"Class {cls}: {count}")


Sample counts after augmentation (first 10):
Class 0: 27
Class 1: 8
Class 2: 11
Class 3: 10
Class 4: 11
Class 5: 12
Class 6: 24
Class 7: 13
Class 8: 10
Class 9: 13


In [ ]:
# Step 9.1 - Build fresh model for augmented training

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout, BatchNormalization

num_classes = len(classes)

model_aug = Sequential([
    # First BiLSTM layer
    Bidirectional(LSTM(128, return_sequences=True), input_shape=(40, 63)),
    BatchNormalization(),
    Dropout(0.3),

    # Second BiLSTM layer
    Bidirectional(LSTM(64)),
    BatchNormalization(),
    Dropout(0.3),

    # Dense layer
    Dense(128, activation='relu'),
    Dropout(0.3),

    # Output layer
    Dense(num_classes, activation='softmax')
])

# Compile model
model_aug.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Show summary
model_aug.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/bidirectional.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_8 (Bidirectional) │ (None, 40, 256)        │       196,608 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 40, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_9 (Bidirectional) │ (None, 128)            │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 88)             │        11,352 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 390,360 (1.49 MB)

 Trainable params: 389,592 (1.49 MB)

 Non-trainable params: 768 (3.00 KB)

In [ ]:
# Step 9.2 - Create callbacks

from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks_aug = [
    EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=4,
        verbose=1
    ),
    ModelCheckpoint(
        "/content/drive/MyDrive/SignNumberModel/best_sign_model_augmented.keras",
        monitor='val_accuracy',
        save_best_only=True
    )
]

In [ ]:
# Step 9.3 - Train model on augmented data

history_aug = model_aug.fit(
    X_aug, y_aug,
    validation_data=(X_val_n, y_val_n),
    epochs=60,
    batch_size=16,
    callbacks=callbacks_aug
)

Epoch 1/60
71/71 ━━━━━━━━━━━━━━━━━━━━ 24s 154ms/step - accuracy: 0.0485 - loss: 4.4867 - val_accuracy: 0.0553 - val_loss: 4.4054 - learning_rate: 5.0000e-04
Epoch 2/60
71/71 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accuracy: 0.1534 - loss: 3.7795 - val_accuracy: 0.0645 - val_loss: 4.1915 - learning_rate: 5.0000e-04
Epoch 3/60
71/71 ━━━━━━━━━━━━━━━━━━━━ 10s 141ms/step - accuracy: 0.2037 - loss: 3.3549 - val_accuracy: 0.1382 - val_loss: 3.8604 - learning_rate: 5.0000e-04
Epoch 4/60
71/71 ━━━━━━━━━━━━━━━━━━━━ 9s 122ms/step - accuracy: 0.2743 - loss: 3.0170 - val_accuracy: 0.1889 - val_loss: 3.3715 - learning_rate: 5.0000e-04
Epoch 5/60
71/71 ━━━━━━━━━━━━━━━━━━━━ 9s 129ms/step - accuracy: 0.3095 - loss: 2.6804 - val_accuracy: 0.2304 - val_loss: 3.0292 - learning_rate: 5.0000e-04
Epoch 6/60
71/71 ━━━━━━━━━━━━━━━━━━━━ 10s 140ms/step - accuracy: 0.3942 - loss: 2.3567 - val_accuracy: 0.3456 - val_loss: 2.6592 - learning_rate: 5.0000e-04
Epoch 7/60
71/71 ━━━━━━━━━━━━━━━━━━━━ 8s 107ms/step - accurac

In [ ]:
# Step 10.1 - Evaluate augmented model

val_loss_aug, val_acc_aug = model_aug.evaluate(X_val_n, y_val_n)

print("Final Validation Accuracy (Augmented Model):", val_acc_aug)

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 170ms/step - accuracy: 0.8018 - loss: 0.7229
Final Validation Accuracy (Augmented Model): 0.8018433451652527


In [ ]:
# Step 10.2 - Predict

y_pred_aug = model_aug.predict(X_val_n)
y_pred_aug = np.argmax(y_pred_aug, axis=1)

print("Prediction shape:", y_pred_aug.shape)

7/7 ━━━━━━━━━━━━━━━━━━━━ 5s 494ms/step
Prediction shape: (217,)


In [ ]:
# Step 10.3 - Manual accuracy

from sklearn.metrics import accuracy_score

acc_aug = accuracy_score(y_val_n, y_pred_aug)
print("Manual Accuracy:", acc_aug)

Manual Accuracy: 0.8018433179723502


In [ ]:
# Step 10.4 - Classification report

from sklearn.metrics import classification_report

print(classification_report(y_val_n, y_pred_aug))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.67      1.00      0.80         2
           2       1.00      0.67      0.80         3
           3       1.00      1.00      1.00         3
           4       0.75      1.00      0.86         3
           5       1.00      0.67      0.80         3
           6       0.00      0.00      0.00         2
           7       0.75      1.00      0.86         3
           8       1.00      0.50      0.67         2
           9       1.00      1.00      1.00         3
          10       1.00      1.00      1.00         2
          11       1.00      0.67      0.80         3
          12       0.67      0.67      0.67         3
          13       0.50      0.50      0.50         2
          14       0.67      0.67      0.67         3
          15       1.00      0.50      0.67         2
          16       0.75      1.00      0.86         3
          17       1.00    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
# Save final model
model_aug.save("/content/drive/MyDrive/SignNumberModel/final_sign_model.keras")

In [ ]:
# Step 1 - Convert BiLSTM model to TFLite using Select TF Ops
# This fixes common conversion errors with LSTM / BiLSTM layers

import tensorflow as tf

# Create converter from trained Keras model
converter = tf.lite.TFLiteConverter.from_keras_model(model_aug)

# Allow both normal TFLite ops and Select TF Ops
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

# Do not lower tensor list ops
converter._experimental_lower_tensor_list_ops = False

# Convert model
tflite_model = converter.convert()

# Save TFLite file
with open("/content/drive/MyDrive/SignNumberModel/final_sign_model_fixed.tflite", "wb") as f:
    f.write(tflite_model)

print("TFLite model saved successfully!")

Saved artifact at '/tmp/tmpfrmj8gcn'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 40, 63), dtype=tf.float32, name='keras_tensor_44')
Output Type:
  TensorSpec(shape=(None, 88), dtype=tf.float32, name=None)
Captures:
  139803235826960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803235825232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219779728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219776656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219772048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219781264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219779536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219780880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219781648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219781072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139803219771472

In [ ]:
# Test TFLite model with one validation sample

import numpy as np
import tensorflow as tf

# Load TFLite model
interpreter = tf.lite.Interpreter(
    model_path="/content/drive/MyDrive/SignNumberModel/final_sign_model_fixed.tflite"
)
interpreter.allocate_tensors()

# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:", input_details)
print("Output details:", output_details)

# Take one sample from validation set
sample = X_val_n[0].astype(np.float32)
sample = np.expand_dims(sample, axis=0)   # make shape (1, 40, 63)

# Set input tensor
interpreter.set_tensor(input_details[0]['index'], sample)

# Run inference
interpreter.invoke()

# Get output
output = interpreter.get_tensor(output_details[0]['index'])
predicted_class = np.argmax(output)

print("Predicted class:", predicted_class)
print("True class:", y_val_n[0])
print("Probabilities shape:", output.shape)

Input details: [{'name': 'serving_default_keras_tensor_44:0', 'index': 0, 'shape': array([ 1, 40, 63], dtype=int32), 'shape_signature': array([-1, 40, 63], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'StatefulPartitionedCall_1:0', 'index': 74, 'shape': array([ 1, 88], dtype=int32), 'shape_signature': array([-1, 88], dtype=int32), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Predicted class: 30
True class: 30
Probabilities shape: (1, 88)


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
# Save label map for mobile app use
import json

# label_map already has: class_name -> index
# we need reverse map: index -> class_name
index_to_label = {v: k for k, v in label_map.items()}

with open("/content/drive/MyDrive/SignNumberModel/index_to_label.json", "w") as f:
    json.dump(index_to_label, f)

print("Label map saved successfully!")
print(index_to_label)

Label map saved successfully!
{0: '1', 1: '10', 2: '100', 3: '103', 4: '105', 5: '109', 6: '11', 7: '110', 8: '12', 9: '124', 10: '13', 11: '130', 12: '135', 13: '14', 14: '140', 15: '15', 16: '150', 17: '16', 18: '160', 19: '165', 20: '17', 21: '170', 22: '18', 23: '19', 24: '190', 25: '199', 26: '2', 27: '20', 28: '200', 29: '210', 30: '220', 31: '230', 32: '24', 33: '25', 34: '26', 35: '28', 36: '3', 37: '30', 38: '32', 39: '34', 40: '35', 41: '36', 42: '38', 43: '4', 44: '40', 45: '45', 46: '5', 47: '50', 48: '52', 49: '54', 50: '55', 51: '56', 52: '58', 53: '6', 54: '60', 55: '62', 56: '64', 57: '66', 58: '68', 59: '69', 60: '7', 61: '70', 62: '72', 63: '74', 64: '75', 65: '76', 66: '78', 67: '8', 68: '80', 69: '83', 70: '84', 71: '85', 72: '88', 73: '9', 74: '90', 75: '92', 76: '94', 77: '95', 78: '96', 79: '97', 80: '98', 81: '99', 82: 'danta-ja na', 83: 'murda-ja na', 84: 'sa', 85: 'saa', 86: 'sha', 87: 'shaa'}


In [ ]:
# Step 1 - Register Python function to receive video from JavaScript
# This saves the recorded webcam video into a file in Colab

from google.colab import output
from base64 import b64decode

def save_video(data):
    # Remove the base64 header part
    video_data = data.split(",")[1]

    # Decode base64 to binary
    binary = b64decode(video_data)

    # Save video file
    with open("live_test_video.webm", "wb") as f:
        f.write(binary)

    print("Video saved as live_test_video.webm")

# Register callback name used by JavaScript
output.register_callback("notebook.save_video", save_video)

print("Python callback registered successfully.")

Python callback registered successfully.


In [3]:
from IPython.display import Javascript, display

display(Javascript("""
(async () => {

const video = document.createElement("video");
video.autoplay = true;
video.width = 320;

const btn = document.createElement("button");
btn.innerText = "🎥 Record 4s";

const status = document.createElement("div");

document.body.appendChild(video);
document.body.appendChild(btn);
document.body.appendChild(status);

let stream = await navigator.mediaDevices.getUserMedia({video:true});
video.srcObject = stream;

status.innerText = "Preview ready. Click Record.";

btn.onclick = async () => {

status.innerText = "Recording 4 seconds...";

const recorder = new MediaRecorder(stream);
let chunks = [];

recorder.ondataavailable = e => chunks.push(e.data);

recorder.start();

await new Promise(r => setTimeout(r,4000));

recorder.stop();

await new Promise(r => recorder.onstop = r);

let blob = new Blob(chunks,{type:"video/webm"});

let reader = new FileReader();
reader.readAsDataURL(blob);

reader.onloadend = () => {

google.colab.kernel.invokeFunction(
"notebook.save_video",
[reader.result],
{}
);

};

status.innerText = "Recording finished and sent to Python.";

};

})();
"""))

<IPython.core.display.Javascript object>

In [4]:
# Step 1 - Set recorded video path
# This is the file saved by your JavaScript recorder

video_path = "live_test_video.webm"
print("Using video:", video_path)

Using video: live_test_video.webm


In [5]:
# Step 2 - Read recorded video and extract landmarks
# This converts video -> sequence of hand landmarks

import cv2
import numpy as np

cap = cv2.VideoCapture(video_path)

sequence = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Extract 63 features from one frame
    landmarks = extract_landmarks(frame)
    sequence.append(landmarks)

cap.release()

sequence = np.array(sequence)

print("Original sequence shape:", sequence.shape)

Original sequence shape: (0,)


In [6]:
# Step 3 - Check if video has frames
# If no frames, stop here

if len(sequence) == 0:
    print("No frames found in the recorded video.")
else:
    print("Video read successfully.")

No frames found in the recorded video.


In [9]:
# Step 4 - Resize sequence to 40 frames
# Model expects (40, 63)

sequence = resize_sequence(sequence)

print("After resize:", sequence.shape)

NameError: name 'resize_sequence' is not defined

In [940]:
# Step 5 - Normalize sequence
# Use the same normalization used in training

sequence = normalize_sequence(sequence)

print("After normalization:", sequence.shape)

After normalization: (40, 63)


In [941]:
# Step 6 - Add batch dimension
# Model input must be (1, 40, 63)

input_seq = np.expand_dims(sequence, axis=0).astype(np.float32)

print("Final input shape:", input_seq.shape)

Final input shape: (1, 40, 63)


In [942]:
# Step 7 - Predict using TFLite model

interpreter.set_tensor(input_details[0]['index'], input_seq)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]['index'])
predicted_class = int(np.argmax(output))
confidence = float(np.max(output))

print("Predicted class index:", predicted_class)
print("Predicted label:", index_to_label[predicted_class])
print("Confidence:", confidence)

Predicted class index: 14
Predicted label: 140
Confidence: 0.24089941382408142


In [875]:
# Step 8 - Show top 5 predictions
# This helps you understand model output better

top5_idx = np.argsort(output[0])[::-1][:5]

print("Top 5 predictions:")
for idx in top5_idx:
    print(f"Class index: {idx}, Label: {index_to_label[int(idx)]}, Score: {output[0][idx]:.4f}")

Top 5 predictions:
Class index: 28, Label: 200, Score: 0.9703
Class index: 2, Label: 100, Score: 0.0290
Class index: 29, Label: 210, Score: 0.0002
Class index: 13, Label: 14, Score: 0.0001
Class index: 1, Label: 10, Score: 0.0001


In [943]:
input_seq.shape = (1, 40, 63)

In [944]:
interpreter.set_tensor(input_details[0]['index'], input_seq)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
predicted_class = np.argmax(output)
predicted_label = index_to_label[predicted_class]

In [945]:
from google.colab import files

files.download("/content/drive/MyDrive/SignNumberModel/final_sign_model_fixed.tflite")
files.download("/content/drive/MyDrive/SignNumberModel/index_to_label.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>